# Agentic RAG with LangChain, Groq, and Tavily

This notebook demonstrates how to build an agent that can use tools (Search) and RAG.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain_community.embeddings import FakeEmbeddings

## Initialize Components

In [ ]:
# LLM
llm = ChatGroq(
    temperature=0,
    model_name="llama3-70b-8192",
    api_key=os.environ.get("GROQ_API_KEY")
)

# Search Tool
search = TavilySearchResults(max_results=2)

# RAG / Retriever Logic
docs = [
    Document(page_content="The Agentic RAG Streamlit App is a daily challenge to build a tool-using agent.", metadata={"source": "challenge_description"}),
    Document(page_content="This app uses Groq for the LLM, Tavily for search, and Streamlit for the UI.", metadata={"source": "tech_stack"}),
    Document(page_content="You are Antigravity, a powerful agentic AI coding assistant.", metadata={"source": "identity"}),
]

embeddings = FakeEmbeddings(size=1536) 
vector = FAISS.from_documents(docs, embeddings)
retriever = vector.as_retriever()

from langchain.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    "search_local_knowledge",
    "Searches for information about the Agentic RAG Streamlit App challenge specifics and identity."
)

tools = [search, retriever_tool]

## Create Agent

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the tools available to answer the user's question."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
# Test the agent
response = agent_executor.invoke({"input": "What tools are used in this project?"})
print(response["output"])